In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import poisson

df_s = pd.read_csv("../results/live_run/shazam_raw.csv")

# FP-Scores = Scores inkorrekt matchender Songs (wie bei Wang)
# Wang: "histogram of scores of incorrectly-matching tracks"
fp_scores = df_s[df_s['result_class'] == 'FP']['score']

# λ schätzen — Erwartungswert der Zufallskollosionen
lambda_est = fp_scores.mean()
print(f"Geschätztes λ: {lambda_est:.2f}")

# Threshold bei gewähltem FPR-Niveau (Wang nennt 0.1% oder 0.01%)
for fpr_target in [0.001, 0.01, 0.05]:
    thresh = poisson.ppf(1 - fpr_target, lambda_est)
    actual_fpr = (fp_scores >= thresh).sum() / len(fp_scores)
    print(f"FPR-Ziel {fpr_target*100:.1f}%: Threshold={thresh:.0f}, "
          f"empirischer FPR={actual_fpr:.4f}")

In [ ]:
import pandas as pd
import numpy as np

df_s = pd.read_csv("../results/live_run/shazam_raw.csv")
fp_scores = df_s[df_s['result_class'] == 'FP']['score']
tp_scores = df_s[df_s['predicted_id'].notna() & 
                  (df_s['result_class'] == 'TP')]['score']

# Empirischen Threshold bei Ziel-FPR bestimmen
# Wang-Methodik, aber direkt auf empirischer Verteilung statt Poisson
for fpr_target in [0.01, 0.05, 0.10]:
    thresh = int(np.percentile(fp_scores, (1 - fpr_target) * 100))
    emp_fpr = (fp_scores >= thresh).sum() / len(fp_scores)
    emp_fnr = (tp_scores < thresh).sum() / len(tp_scores)
    print(f"FPR-Ziel {fpr_target*100:.0f}%: T={thresh}, "
          f"FPR={emp_fpr:.4f}, FNR={emp_fnr:.4f}")

In [ ]:
import pandas as pd
import numpy as np

df_s = pd.read_csv("/home/jupyter/testrun/results/dry_run/shazam_raw.csv")
fp_scores = df_s[df_s['result_class'] == 'FP']['score']
tp_scores = df_s[df_s['predicted_id'].notna() & 
                  (df_s['result_class'] == 'TP')]['score']

# Empirischen Threshold bei Ziel-FPR bestimmen
# Wang-Methodik, aber direkt auf empirischer Verteilung statt Poisson
for fpr_target in [0.01, 0.05, 0.10]:
    thresh = int(np.percentile(fp_scores, (1 - fpr_target) * 100))
    emp_fpr = (fp_scores >= thresh).sum() / len(fp_scores)
    emp_fnr = (tp_scores < thresh).sum() / len(tp_scores)
    print(f"FPR-Ziel {fpr_target*100:.0f}%: T={thresh}, "
          f"FPR={emp_fpr:.4f}, FNR={emp_fnr:.4f}")

In [ ]:
import librosa
import glob

# Stichprobe: 5 Query-Dateien verschiedener Conditions
paths = glob.glob("../data/queries/live_run/*.wav")[:5]
for p in paths:
    y, sr = librosa.load(p, sr=None)  # sr=None = keine Resampling, nativen Wert lesen
    print(f"{sr} Hz | {len(y)/sr:.1f}s | {p.split('/')[-1]}")